## Kerendia Data Sources - Date Profiling
For every Snowflake table in `Kerendia_Data_Sources.md`, returns:
- **Service date column** used (auto-detected by priority list, else first DATE/TIMESTAMP column)
- **Min / Max Service Date**
- **Last Updated** (`INFORMATION_SCHEMA.TABLES.LAST_ALTERED`; for views this reflects DDL change, not data refresh)

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
# (S.No in md, db, schema, table)
TABLES = [
    (1,  "PHCDW", "PHCDW_STG_CAR", "STG_CAR_IQV_LAAD_FACT_DX"),
    (2,  "PHCDW", "PHCDW_STG_CAR", "STG_CAR_IQV_LAAD_FACT_PX"),
    (3,  "PHCDW", "PHCDW_STG_CAR", "STG_CAR_IQV_LAAD_FCT_RX"),
    (4,  "PHCDW", "PHCDW_DSAA", "KERENDIA_LAAD_BASED_LRX_V6"),
    (5,  "PHCDW", "PHCDW_STG_CAR", "STG_CAR_IQV_QUEST_TO_LAAD_HIST"),
    (6,  "PHCDW", "PHCDW_SANDBOX", "VIEW_RB_KERENDIA_COPAY_PROGRAM"),
    (7,  "CPH_DB_PROD", "ENRICH_V2", "MMX_FLASHTALKING_NEW_DAILYFILES_V1"),
    (7,  "CPH_DB_PROD", "ENRICH", "FLASHTALKING_DMA"),
    (8,  "PHCDW", "PHCDW_CDM", "MMX_FNL_COMBINED_AGGR_TAB"),
    (9,  "CPH_DB_PROD", "MODEL_V2", "FACT_CALL"),
    (9,  "CPH_DB_PROD", "ANALYTICS_V2", "ANLT_BASE_FACT_CUST_CALL_ACTY"),
    (10, "PHCDW", "PHCDW_SANDBOX", "VIEW_RB_KERENDIA_VOUCHER_FTP"),
    (11, "FUNNEL_PROD_INB2", "FUNNEL_MK7FZ_CN_UWQY_VMROR", "FUNNEL_DATA_FACEBOOKADS_GEO"),
    (12, "CPH_DB_PROD", "ANALYTICS_V2", "ANLT_BASE_FACT_KERENDIA_SAMPLES_SHIPPED_DROPPED_KER"),
    (12, "CPH_DB_PROD", "ANALYTICS_V2", "KERENDIA_SAMPLES"),
    (12, "CPH_DB_PROD", "ANALYTICS_V2", "KERENDIA_SAMPLES_SHIPPED"),
    (13, "CPH_DB_PROD", "ANALYTICS_V2", "ANLT_BASE_FACT_EVNT_PRSNL_PROM"),
    (14, "CPH_DB_PROD", "MODEL_V2", "FACT_SENT_EMAIL"),
    (15, "CPH_DB_PROD", "ANALYTICS_V2", "ANLT_BASE_FACT_CLKSTRM"),
    (16, "CPH_DB_PROD", "ANALYTICS_V2", "ANLT_DRG_FRMLRY_HIST"),
    (17, "CPH_DB_PROD", "MODEL_V2", "FACT_CALL"),
    (17, "CPH_DB_PROD", "ANALYTICS", "FACT_CALL"),
    (18, "PHCDW", "PHCDW_STG", "MDM_STG_HCP_ADDR"),
    (19, "PHCDW", "PHCDW_TARGETING", "BPT_KER_HCP"),
    (20, "PHCDW", "PHCDW_TARGETING", "CP_KER_HCP"),
    (21, "PHCDW", "PUB", "CUST_MDM_BEST_AFFL_PUBLISH_PD_OUTP_STATIC"),
    (22, "CPH_DB_PROD", "ANALYTICS_V2", "ANLT_BASE_HCP_BRAND_PRFL_KER_UNIV"),
]

# Preferred service-date column names (first match wins)
PRIORITY = ["SVC_DT", "SERVICE_DATE", "SVC_DATE", "CALL_DT", "CALL_DATE", "ACTVY_DT", "ACTY_DT", "EVNT_DT", "EVENT_DATE",
            "EVNT_STRT_DT", "SENT_DT", "SENT_DATE", "EMAIL_SENT_DT", "SHIP_DT", "SHIPPED_DATE", "DROP_DT", "TRANSACTION_DATE",
            "TXN_DT", "DATE", "DAY", "WEEK_ENDING_DATE", "WEEK_END_DT", "DATA_DATE", "REPORT_DATE", "MONTH", "PERIOD_ID"]

In [0]:
import pandas as pd

rows = []
for sno, db, sch, tbl in TABLES:
    fq = f"{db}.{sch}.{tbl}"
    rec = {"S_No": sno, "Table": fq, "Date_Col": None, "Min_Service_Date": None, "Max_Service_Date": None,
           "Last_Updated": None, "Table_Type": None, "Error": None}
    try:
        m = Get_Data_Snowflakes(f"""SELECT TABLE_TYPE, TO_VARCHAR(LAST_ALTERED, 'YYYY-MM-DD') AS LAST_ALTERED
                                   FROM {db}.INFORMATION_SCHEMA.TABLES
                                   WHERE TABLE_SCHEMA = '{sch}' AND TABLE_NAME = '{tbl}'""").collect()
        if m:
            rec["Table_Type"], rec["Last_Updated"] = m[0]["TABLE_TYPE"], m[0]["LAST_ALTERED"]
        cols = Get_Data_Snowflakes(f"""SELECT COLUMN_NAME, DATA_TYPE FROM {db}.INFORMATION_SCHEMA.COLUMNS
                                      WHERE TABLE_SCHEMA = '{sch}' AND TABLE_NAME = '{tbl}'
                                      ORDER BY ORDINAL_POSITION""").toPandas()
        date_cols = cols[cols.DATA_TYPE.str.startswith(("DATE", "TIMESTAMP"))].COLUMN_NAME.tolist()
        pick = next((c for c in PRIORITY if c in date_cols), None) or next((c for c in date_cols if "DT" in c or "DATE" in c), None) or (date_cols[0] if date_cols else None)
        if pick is None:  # fall back to string/number columns that look like dates
            pick = next((c for c in PRIORITY if c in cols.COLUMN_NAME.tolist()), None)
        rec["Date_Col"] = pick
        if pick:
            r = Get_Data_Snowflakes(f'SELECT TO_VARCHAR(MIN("{pick}")) AS MN, TO_VARCHAR(MAX("{pick}")) AS MX FROM {fq}').collect()[0]
            rec["Min_Service_Date"], rec["Max_Service_Date"] = r["MN"], r["MX"]
    except Exception as e:
        rec["Error"] = str(e)[:200]
    rows.append(rec)
    print(rec)

result = pd.DataFrame(rows)
display(result)

In [0]:
result.to_csv("/Workspace/Users/saurabh.modi.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/02_Input_Data_Creation/00_Data_Migration/data_sources_date_profile.csv", index=False)

In [0]:
CHECK = [("PHCDW","PHCDW_STG_CAR","STG_CAR_IQV_LAAD_FACT_PX"), ("PHCDW","PHCDW_STG_CAR","STG_CAR_IQV_QUEST_TO_LAAD_HIST"),
         ("CPH_DB_PROD","MODEL_V2","FACT_CALL"), ("CPH_DB_PROD","ANALYTICS_V2","ANLT_BASE_FACT_CLKSTRM"),
         ("CPH_DB_PROD","ANALYTICS_V2","ANLT_DRG_FRMLRY_HIST"), ("CPH_DB_PROD","ANALYTICS_V2","KERENDIA_SAMPLES_SHIPPED"),
         ("PHCDW","PHCDW_TARGETING","BPT_KER_HCP"), ("PHCDW","PHCDW_TARGETING","CP_KER_HCP"),
         ("CPH_DB_PROD","ANALYTICS_V2","ANLT_BASE_HCP_BRAND_PRFL_KER_UNIV"), ("CPH_DB_PROD","MODEL_V2","FACT_SENT_EMAIL"),
         ("PHCDW","PHCDW_STG","MDM_STG_HCP_ADDR"), ("PHCDW","PUB","CUST_MDM_BEST_AFFL_PUBLISH_PD_OUTP_STATIC")]
for db, sch, tbl in CHECK:
    c = Get_Data_Snowflakes(f"""SELECT COLUMN_NAME, DATA_TYPE FROM {db}.INFORMATION_SCHEMA.COLUMNS
          WHERE TABLE_SCHEMA='{sch}' AND TABLE_NAME='{tbl}'
          AND (DATA_TYPE LIKE 'DATE%' OR DATA_TYPE LIKE 'TIMESTAMP%' OR COLUMN_NAME ILIKE '%DT%' OR COLUMN_NAME ILIKE '%DATE%'
               OR COLUMN_NAME ILIKE '%MONTH%' OR COLUMN_NAME ILIKE '%WEEK%' OR COLUMN_NAME ILIKE '%PERIOD%' OR COLUMN_NAME ILIKE '%QTR%' OR COLUMN_NAME ILIKE '%YEAR%')
          ORDER BY ORDINAL_POSITION""").toPandas()
    print(f"{db}.{sch}.{tbl}:", ", ".join(f"{a}({b})" for a, b in zip(c.COLUMN_NAME, c.DATA_TYPE)))

# FUNNEL table returned no metadata - check visibility
for q in ["SELECT COUNT(*) AS N FROM FUNNEL_PROD_INB2.FUNNEL_MK7FZ_CN_UWQY_VMROR.FUNNEL_DATA_FACEBOOKADS_GEO",
          "SELECT COLUMN_NAME, DATA_TYPE FROM FUNNEL_PROD_INB2.INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME='FUNNEL_DATA_FACEBOOKADS_GEO'"]:
    try:
        print(Get_Data_Snowflakes(q).toPandas().to_string())
    except Exception as e:
        print("FUNNEL ERR:", str(e)[:300])

In [0]:
# Override: correct business/service date column per table (TEXT columns parsed with TRY_TO_DATE)
OVERRIDE = [
    ("PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX", "SERVICE_DATE"),
    ("PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST", "SVC_DT"),
    ("CPH_DB_PROD.MODEL_V2.FACT_CALL", "CALL_DATE"),
    ("CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_CLKSTRM", "DATE_ID"),
    ("CPH_DB_PROD.ANALYTICS_V2.ANLT_DRG_FRMLRY_HIST", "DATA_MONTH"),
    ("CPH_DB_PROD.ANALYTICS_V2.KERENDIA_SAMPLES_SHIPPED", "SMPLE_ORDR_TRNS_CLL_DT"),
    ("PHCDW.PHCDW_TARGETING.CP_KER_HCP", "CALL_PLAN_START_DATE"),
    ("PHCDW.PHCDW_TARGETING.CP_KER_HCP", "CALL_PLAN_END_DATE"),
    ("CPH_DB_PROD.MODEL_V2.FACT_SENT_EMAIL", "EMAIL_SNT_DATE"),
    ("CPH_DB_PROD.ANALYTICS.FACT_CALL", "CALL_DATE"),
    ("CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_EVNT_PRSNL_PROM", "EVNT_START_DATE"),
]
for fq, colname in OVERRIDE:
    try:
        r = Get_Data_Snowflakes(f'''SELECT MIN("{colname}")::VARCHAR AS RAW_MIN, MAX("{colname}")::VARCHAR AS RAW_MAX,
              MIN(TRY_TO_DATE("{colname}"::VARCHAR))::VARCHAR AS MN, MAX(TRY_TO_DATE("{colname}"::VARCHAR))::VARCHAR AS MX,
              MAX(CASE WHEN TRY_TO_DATE("{colname}"::VARCHAR) <= CURRENT_DATE THEN TRY_TO_DATE("{colname}"::VARCHAR) END)::VARCHAR AS MX_LE_TODAY
              FROM {fq}''').collect()[0]
        print(fq, colname, dict(r.asDict()))
    except Exception as e:
        print(fq, colname, "ERR", str(e)[:200])

f = Get_Data_Snowflakes("""SELECT COLUMN_NAME, DATA_TYPE FROM FUNNEL_PROD_INB2.INFORMATION_SCHEMA.COLUMNS
      WHERE TABLE_NAME='FUNNEL_DATA_FACEBOOKADS_GEO' AND (DATA_TYPE LIKE 'DATE%' OR DATA_TYPE LIKE 'TIMESTAMP%' OR COLUMN_NAME ILIKE '%DATE%' OR COLUMN_NAME ILIKE '%DAY%')""").toPandas()
print(f.to_string())
print(Get_Data_Snowflakes("SELECT TABLE_SCHEMA, TABLE_TYPE, LAST_ALTERED::VARCHAR AS LAST_ALTERED FROM FUNNEL_PROD_INB2.INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME='FUNNEL_DATA_FACEBOOKADS_GEO'").toPandas().to_string())

In [0]:
Q = {
  "QUEST SVC_DT": "SELECT MIN(TRY_TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY'))::VARCHAR MN, MAX(TRY_TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY'))::VARCHAR MX FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST",
  "CLKSTRM DATE_ID": "SELECT MIN(TRY_TO_DATE(DATE_ID,'YYYYMMDD'))::VARCHAR MN, MAX(TRY_TO_DATE(DATE_ID,'YYYYMMDD'))::VARCHAR MX FROM CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_CLKSTRM",
  "DRG DATA_MONTH": "SELECT MIN(TRY_TO_DATE(DATA_MONTH,'MM/YYYY'))::VARCHAR MN, MAX(TRY_TO_DATE(DATA_MONTH,'MM/YYYY'))::VARCHAR MX FROM CPH_DB_PROD.ANALYTICS_V2.ANLT_DRG_FRMLRY_HIST",
  "FUNNEL DATE": "SELECT MIN(\"DATE\")::VARCHAR MN, MAX(\"DATE\")::VARCHAR MX FROM FUNNEL_PROD_INB2.FUNNEL__MK7FZ_CN_UWQY_VMROR.FUNNEL_DATA_FACEBOOKADS_GEO",
}
for k, q in Q.items():
    try:
        print(k, Get_Data_Snowflakes(q).collect()[0].asDict())
    except Exception as e:
        print(k, "ERR", str(e)[:250])

In [0]:
print(Get_Data_Snowflakes("""SELECT MIN(D)::VARCHAR MN, COUNT_IF(D < '2000-01-01') N_PLACEHOLDER
  FROM (SELECT TRY_TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY') D FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST)
  WHERE D >= '2000-01-01' OR D < '2000-01-01'""").collect()[0].asDict())
print(Get_Data_Snowflakes("""SELECT MIN(D)::VARCHAR MN FROM (SELECT TRY_TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY') D FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST) WHERE D >= '2000-01-01'""").collect()[0].asDict())

In [0]:
SUPPORT = [("MODEL_V2","FACT_IQV_LAAD_SUB_NATIONAL"), ("MODEL_V2","FACT_IQV_LAAD_SUB_NATIONAL_XREF"), ("MODEL_V2","MAP_CUSTOMER_XREF"),
           ("MODEL_V2","DIM_CUSTOMER_HCP"), ("ANALYTICS","DIM_PRODUCT"), ("MODEL_V2","DIM_APPROVED_DOCUMENT")]
inlist = ",".join(f"('{s}','{t}')" for s, t in SUPPORT)
print(Get_Data_Snowflakes(f"""SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE, TO_VARCHAR(LAST_ALTERED,'YYYY-MM-DD') AS LAST_ALTERED
      FROM CPH_DB_PROD.INFORMATION_SCHEMA.TABLES WHERE (TABLE_SCHEMA, TABLE_NAME) IN ({inlist})""").toPandas().to_string())